# Week #6(b): Build an ARC Solver

[Open this notebook in Colab](https://colab.research.google.com/github/neurosynthetic/grant-vogt/blob/main/notebooks/week06b.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

In Week 5(b), you chose transformations and connected them into pipelines. In Week 6(a), you learned how to make random choices. Now you will combine those skills to build a program that **searches for a pipeline on its own**.

The idea is:

> **Propose a sequence → test it on every training example → keep it if it works**

A sequence of transformations is a small program. Searching for such a sequence is a form of **program search**, an idea used in AI research.

By the end, your solver will discover programs for the ARC-style tasks in this notebook and apply them to new inputs. Its abilities depend on its library, its allowed sequence length, and its search budget. Finding programs that generalize to unfamiliar tasks is a much broader research challenge—and this lesson gives you a concrete piece of it to build yourself.

## Goals and roadmap

You will build six pieces:

1. Check whether one named transformation explains an example.
2. Apply a list of transformations in order.
3. Generate a random list of transformation names.
4. Check one list against every training example.
5. Repeatedly propose lists until one matches or the attempt limit is reached.
6. Use the discovered list on a new test input.

Then use your solver on three tasks. An optional fourth task allows longer programs.

All tasks here were created for this lesson; they are not from the official ARC dataset.

## Setup

Run the next two collapsed cells once. The first provides pictures, buttons, and checking tools. The second provides the transformation library and `apply_step(grid, name)`.

You will write the search and matching code. If the runtime restarts, rerun setup and your function cells.

In [ ]:
#@title Setup utilities (provided — run once)

import copy
import random
from contextlib import contextmanager
from unittest.mock import patch

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25",
]
GRID_CMAP = ListedColormap(GRID_COLORS)


def draw_grid(ax, grid, title=""):
    rows, cols = len(grid), len(grid[0])
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9, interpolation="nearest")
    font_size = max(6, min(12, 120 // max(rows, cols)))
    for row_index, row in enumerate(grid):
        for column_index, color in enumerate(row):
            text_color = "white" if color in {0, 1, 2, 6, 9} else "black"
            ax.text(column_index, row_index, str(color), ha="center", va="center",
                    color=text_color, fontsize=font_size, fontweight="bold")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(which="both", left=False, bottom=False, labelleft=False, labelbottom=False)


def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(4, 3))
    draw_grid(ax, grid, title)
    plt.tight_layout()
    plt.show()


def show_pair(input_grid, output_grid, label=""):
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    draw_grid(axes[0], input_grid, "Input")
    draw_grid(axes[1], output_grid, "Output")
    if label:
        fig.suptitle(label)
    fig.subplots_adjust(top=0.78 if label else 0.90, bottom=0.06,
                        left=0.05, right=0.95, wspace=0.18)
    plt.show()


def show_examples(examples):
    for number, example in enumerate(examples, start=1):
        show_pair(example["input"], example["output"], f"Training example {number}")


def run_check(check_fn):
    try:
        check_fn()
    except AssertionError as exc:
        print("Not yet:", exc)
        return False
    except Exception as exc:
        print("Your code raised an error:", exc)
        return False
    print("All checks pass!")
    return True


def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            run_check(check_fn)

    button.on_click(on_click)
    display(button, output)


@contextmanager
def _repeatable_random(seed):
    state = random.getstate()
    try:
        random.seed(seed)
        yield
    finally:
        random.setstate(state)


def run_solver(examples, test_input, max_steps, max_attempts, seed):
    # Set a seed once per search, not once per proposed sequence.
    with _repeatable_random(seed):
        try:
            prediction = solve_task(examples, test_input, max_steps, max_attempts)
            if prediction is None:
                print("No prediction returned. Finish the exercises first; if they pass,")
                print("the search may need a different seed, more attempts, or a longer sequence.")
                return None
            show_pair(test_input, prediction, "Your solver's prediction")
            return prediction
        except Exception as exc:
            print("Your solver raised an error:", exc)
            print("Use the exercise checkers to locate the problem.")
            return None


print("Setup complete.")


In [ ]:
#@title Transformation library (provided — run once)

STEP_NAMES = [
    "flip_horizontal",
    "flip_vertical",
    "transpose",
    "crop",
    "blue_to_red",
    "yellow_border",
]


def flip_horizontal(grid):
    return [row[::-1] for row in grid]


def flip_vertical(grid):
    return [row[:] for row in grid[::-1]]


def transpose_grid(grid):
    output = []
    for column_index in range(len(grid[0])):
        new_row = []
        for row in grid:
            new_row.append(row[column_index])
        output.append(new_row)
    return output


def crop_object(grid):
    occupied_rows = []
    occupied_columns = []
    for row_index in range(len(grid)):
        for column_index in range(len(grid[0])):
            if grid[row_index][column_index] != 0:
                occupied_rows.append(row_index)
                occupied_columns.append(column_index)
    if not occupied_rows:
        return [row[:] for row in grid]
    top = min(occupied_rows)
    bottom = max(occupied_rows)
    left = min(occupied_columns)
    right = max(occupied_columns)
    return [row[left:right + 1] for row in grid[top:bottom + 1]]


def blue_to_red(grid):
    output = []
    for row in grid:
        new_row = []
        for color in row:
            if color == 1:
                new_row.append(2)
            else:
                new_row.append(color)
        output.append(new_row)
    return output


def yellow_border(grid):
    width = len(grid[0]) + 2
    output = [[4] * width]
    for row in grid:
        output.append([4] + row[:] + [4])
    output.append([4] * width)
    return output


def apply_step(grid, name):
    if name == "flip_horizontal":
        return flip_horizontal(grid)
    elif name == "flip_vertical":
        return flip_vertical(grid)
    elif name == "transpose":
        return transpose_grid(grid)
    elif name == "crop":
        return crop_object(grid)
    elif name == "blue_to_red":
        return blue_to_red(grid)
    elif name == "yellow_border":
        return yellow_border(grid)
    raise ValueError("Unknown transformation name: " + str(name))


print("Library ready:", STEP_NAMES)


## Six choices for the solver

Every library function takes a grid and returns a new grid without changing its input. Any color choices are fixed inside the functions.

| Name | Effect |
| --- | --- |
| `"flip_horizontal"` | Reverse the cells in every row |
| `"flip_vertical"` | Reverse the order of rows |
| `"transpose"` | Swap rows and columns |
| `"crop"` | Keep the smallest rectangle containing all nonzero cells |
| `"blue_to_red"` | Change blue (`1`) to red (`2`); keep other colors |
| `"yellow_border"` | Add a one-cell yellow (`4`) border |

For a completely black grid, `"crop"` returns an unchanged copy. These functions can therefore be chained in any order on a nonempty rectangular grid.

The provided `apply_step` helper runs the named function:

```python
changed_grid = apply_step(grid, "flip_horizontal")
```

Quotation marks matter: `"flip_horizontal"` is a string that names an operation. Passing that string to `apply_step` runs the operation.

## Exercise 1: Does one step explain an example?

An ARC training example stores two grids:

```python
example["input"]
example["output"]
```

You used these dictionary keys in earlier lessons. We will only read the examples.

Write `matches_one_step(example, name)`:

1. Apply the named transformation to `example["input"]`.
2. Compare the result with `example["output"]`.
3. Return `True` if they match, and `False` otherwise.

Python can compare complete grids with `==`. It compares the lists and their values, including their order.

In [ ]:
def matches_one_step(example, name):
    return None


In [ ]:
#@title Check Exercise 1

def check_exercise_1():
    tests = [
        ({"input": [[1, 0, 2], [3, 4, 0]],
          "output": [[2, 0, 1], [0, 4, 3]]}, "flip_horizontal", True),
        ({"input": [[1, 0, 2], [3, 4, 0]],
          "output": [[2, 0, 1], [0, 4, 3]]}, "flip_vertical", False),
        ({"input": [[1, 0], [3, 1]],
          "output": [[2, 0], [3, 2]]}, "blue_to_red", True),
        ({"input": [[1, 2, 3]], "output": [[1], [2], [3]]}, "transpose", True),
    ]
    for example, name, expected in tests:
        original = copy.deepcopy(example)
        got = matches_one_step(example, name)
        assert got is expected, f"Check {name}: return the Boolean {expected}."
        assert example == original, "Leave both training grids unchanged."


make_check_button(check_exercise_1)


## Exercise 2: Run a whole sequence

A candidate program is an ordinary list of names:

```python
steps = ["transpose", "flip_horizontal"]
```

Write `apply_steps(grid, steps)`. Start with the input grid. For each name, call `apply_step` on the **current result**, then keep its output for the next step.

Return the final grid. An empty list of steps should return an unchanged grid. Do not change the input grid or the list of names.

This is the same composition idea from Week 5(b), with a loop deciding how many steps to run.

In [ ]:
def apply_steps(grid, steps):
    result = grid
    # Apply each named step to the current result.
    return result


In [ ]:
#@title Check Exercise 2

def check_exercise_2():
    grid = [[1, 2, 3], [4, 5, 6]]
    cases = [
        ([], [[1, 2, 3], [4, 5, 6]]),
        (["flip_horizontal"], [[3, 2, 1], [6, 5, 4]]),
        (["transpose", "flip_horizontal"], [[4, 1], [5, 2], [6, 3]]),
        (["flip_horizontal", "transpose"], [[3, 6], [2, 5], [1, 4]]),
        (["flip_horizontal", "flip_horizontal"], [[1, 2, 3], [4, 5, 6]]),
        (["transpose", "blue_to_red", "flip_vertical"], [[3, 6], [2, 5], [2, 4]]),
    ]
    for steps, expected in cases:
        original_grid = copy.deepcopy(grid)
        original_steps = steps[:]
        got = apply_steps(grid, steps)
        assert got == expected, f"Check the order and intermediate results for {steps}."
        assert grid == original_grid, "The input grid changed."
        assert steps == original_steps, "The list of step names changed."


make_check_button(check_exercise_2)


## A new random tool: `random.choice`

Week 6(a) introduced `random.randint`. We also mentioned `random.choice` as an optional experiment; here it becomes part of the main lesson.

`random.choice(items)` chooses one item from a nonempty list. Each position in that list has the same chance of being selected.

```python
name = random.choice(STEP_NAMES)
```

This chooses a transformation **name**. It does not apply the transformation yet. Run the next cell a few times. Repeated names are normal.

In [ ]:
for choice_number in range(5):
    print(random.choice(STEP_NAMES))


## Exercise 3: Generate a random program

Write `random_steps(max_steps)`. It should:

1. Choose a length from `1` through `max_steps`, using `random.randint`.
2. Create an empty list.
3. Choose that many names with `random.choice(STEP_NAMES)`, appending each one.
4. Return the list.

Choose each name separately. Repeated names are allowed: two horizontal flips, for example, undo each other.

You may assume `max_steps` is a positive integer. Do not reset the seed inside this function; each call should make fresh choices.

In [ ]:
def random_steps(max_steps):
    return None


In [ ]:
#@title Check Exercise 3

def check_exercise_3():
    original_names = STEP_NAMES[:]
    with _repeatable_random(6203):
        for limit in (1, 2, 3):
            samples = [random_steps(limit) for _ in range(180)]
            assert all(type(steps) is list for steps in samples), "Return a list of names."
            assert all(1 <= len(steps) <= limit for steps in samples), "Check the length limits."
            assert all(name in STEP_NAMES for steps in samples for name in steps), "Use library names."
            assert {len(steps) for steps in samples} == set(range(1, limit + 1)), (
                "Choose the length randomly, including both endpoints."
            )
            assert {name for steps in samples for name in steps} == set(STEP_NAMES), (
                "Use random.choice to give every library name a chance."
            )
            if limit > 1:
                assert any(len(set(steps)) < len(steps) for steps in samples), (
                    "Repeated names must be allowed."
                )
    assert STEP_NAMES == original_names, "Leave the provided library-name list unchanged."


make_check_button(check_exercise_3)


## Exercise 4: One program must explain every example

Write `matches_examples(steps, examples)`. You may assume `examples` contains at least one training pair.

For each example:

- apply the **same** `steps` to that example's input
- compare the result with its expected output
- return `False` immediately if there is a mismatch

Return `True` only after every example has passed.

Use `!=` to test whether two grids differ.

Use `apply_steps` here. Do not choose new random steps inside this function. Each example starts from its own input grid, and the candidate program stays fixed throughout the check.

In [ ]:
def matches_examples(steps, examples):
    return None


In [ ]:
#@title Check Exercise 4

def check_exercise_4():
    examples = [
        {"input": [[1, 0], [0, 1]], "output": [[0, 1], [1, 0]]},
        {"input": [[1, 0, 2], [3, 0, 0]], "output": [[2, 0, 1], [0, 0, 3]]},
    ]
    for pairs in (examples, examples[::-1]):
        for steps, expected in [
            (["flip_horizontal"], True),
            (["flip_vertical"], False),
            ([], False),
            (["transpose", "transpose", "flip_horizontal"], True),
        ]:
            original_pairs = copy.deepcopy(pairs)
            original_steps = steps[:]
            got = matches_examples(steps, pairs)
            assert got is expected, (
                f"Check every example for {steps}; return {expected}. "
                "Do not return True from inside the example loop."
            )
            assert pairs == original_pairs, "Training examples changed."
            assert steps == original_steps, "The candidate sequence changed."


make_check_button(check_exercise_4)


## Exercise 5: Search for a matching program

Write `find_solution(examples, max_steps, max_attempts)`.

Use a loop that runs at most `max_attempts` times. On each attempt:

1. Call `random_steps(max_steps)` to propose one candidate.
2. Call `matches_examples(candidate, examples)` to test it.
3. If it matches, immediately return that list of names.

If the loop finishes without a match, return `None`. Here, `None` means **“this search did not find a solution.”** It is different from an empty list of steps.

You may assume `max_steps >= 1` and `max_attempts >= 0`. With zero attempts, return `None` without generating a candidate.

The attempt limit keeps the program from searching forever. Random choices can repeat, so even many attempts might miss a useful candidate. A matching program also need not be the shortest one.

In [ ]:
def find_solution(examples, max_steps, max_attempts):
    return None


In [ ]:
#@title Check Exercise 5

def check_exercise_5():
    examples = [
        {"input": [[1, 0], [0, 1]], "output": [[0, 1], [1, 0]]},
        {"input": [[1, 0, 2], [3, 0, 0]], "output": [[2, 0, 1], [0, 0, 3]]},
    ]
    # Controlled candidate streams test the stopping rule without relying on luck.
    scenarios = [
        ([["flip_vertical"], ["flip_horizontal"]], 5, ["flip_horizontal"], 2),
        ([["flip_horizontal"]], 5, ["flip_horizontal"], 1),
        ([["flip_vertical"], ["transpose"]], 2, None, 2),
        ([], 0, None, 0),
    ]
    for candidates, budget, expected, expected_calls in scenarios:
        calls = []

        def fake_random_steps(limit):
            assert limit == 2, "Pass max_steps to random_steps."
            index = len(calls)
            calls.append(limit)
            assert index < len(candidates), "Stop when a match is found or the budget runs out."
            return candidates[index][:]

        original = copy.deepcopy(examples)
        with patch.dict(globals(), {"random_steps": fake_random_steps}):
            got = find_solution(examples, 2, budget)
        assert got == expected, (
            "Return the first sequence that matches every example, or None after the limit."
        )
        assert len(calls) == expected_calls, "Check the number of attempts and when you return."
        assert examples == original, "Do not change the evidence while searching."

    # Also exercise the real random generator and matching functions together.
    with _repeatable_random(6205):
        got = find_solution(examples, 2, 1000)
    assert type(got) is list and 1 <= len(got) <= 2, "The real search did not return a candidate."
    assert all(name in STEP_NAMES for name in got), "Return transformation names."
    assert all(_apply_for_check(e["input"], got) == e["output"] for e in examples), (
        "The discovered sequence must match all training pairs."
    )


def _apply_for_check(grid, steps):
    # Uses only the provided library, independent of the student's apply_steps.
    result = copy.deepcopy(grid)
    for name in steps:
        result = apply_step(result, name)
    return result


make_check_button(check_exercise_5)


## Matching examples is evidence, not certainty

Consider this training pair. Both a horizontal flip and a vertical flip produce its output.

In [ ]:
ambiguous_examples = [
    {
        "input": [
            [1, 0],
            [0, 1],
        ],
        "output": [
            [0, 1],
            [1, 0],
        ],
    },
]

show_examples(ambiguous_examples)
print("Horizontal flip fits:", matches_examples(["flip_horizontal"], ambiguous_examples))
print("Vertical flip fits:  ", matches_examples(["flip_vertical"], ambiguous_examples))


Now add an example with less symmetry. The same two candidates no longer explain the evidence equally well.

In [ ]:
extra_example = {
    "input": [
        [1, 0, 2],
        [3, 0, 0],
    ],
    "output": [
        [2, 0, 1],
        [0, 0, 3],
    ],
}
more_examples = ambiguous_examples + [extra_example]

show_examples([extra_example])
print("Horizontal flip still fits:", matches_examples(["flip_horizontal"], more_examples))
print("Vertical flip still fits:  ", matches_examples(["flip_vertical"], more_examples))


### Think before moving on

- If the solver finds the vertical flip after seeing only the first example, has it made a mistake?
- What changed when the second example was added?
- Could two different lists of steps describe the same transformation?

The solver can only test the evidence it receives. More informative examples can rule out incorrect hypotheses; different programs can also be equivalent.

## Exercise 6: Solve a new input

Write `solve_task(examples, test_input, max_steps, max_attempts)`.

1. Call `find_solution` using the training examples.
2. If the result is `None`, return `None`.
3. Otherwise, print the discovered steps and apply them to `test_input`.
4. Return the predicted output grid.

To check for the special value `None`, write `if steps is None:`. If that condition is true, return `None` before trying to apply any steps.

The test input is **not** another training pair. Its expected output is not given to the search.

The search chooses one program using the training evidence. Applying that program to the new input tests whether it generalizes.

In [ ]:
def solve_task(examples, test_input, max_steps, max_attempts):
    return None


In [ ]:
#@title Check Exercise 6

def check_exercise_6():
    examples = [
        {"input": [[1, 0, 2]], "output": [[2, 0, 1]]},
    ]
    test_input = [[3, 4, 0], [0, 5, 6]]
    original_examples = copy.deepcopy(examples)
    original_test = copy.deepcopy(test_input)

    def fake_find_solution(pairs, max_steps, max_attempts):
        assert pairs == original_examples, "Search using the training examples only."
        assert max_steps == 2 and max_attempts == 25, "Forward the search limits."
        return ["flip_horizontal"]

    with patch.dict(globals(), {"find_solution": fake_find_solution}):
        result = solve_task(examples, test_input, 2, 25)
    assert result == [[0, 4, 3], [6, 5, 0]], "Apply the discovered steps to the new test input."

    with patch.dict(globals(), {"find_solution": lambda *args: None}):
        result = solve_task(examples, test_input, 2, 0)
    assert result is None, "If the search returns None, the solver should return None."
    assert examples == original_examples and test_input == original_test, "Leave inputs unchanged."


make_check_button(check_exercise_6)


# Your solver in action

Finish all six exercises before running these searches. Each task contains two training pairs and one new test input.

For each task:

1. Inspect the examples and predict the test output yourself.
2. Run the provided search cell.
3. Read the sequence your program discovered.
4. Compare your prediction with the generated grid, then click the task's checker.

The search cells set a seed once before each search to make the demonstration reproducible. Checkers compare outputs, so equivalent programs are accepted.

The hidden test answers are used only by the task checkers, after the search has returned.

## Task A

Study both training examples and the test input before starting the search.

In [ ]:
training_examples_a = [
    {
        "input": [
            [1, 0, 2],
            [0, 3, 3]
        ],
        "output": [
            [2, 0, 1],
            [3, 3, 0]
        ],
    },
    {
        "input": [
            [0, 4],
            [5, 4],
            [0, 6]
        ],
        "output": [
            [4, 0],
            [4, 5],
            [6, 0]
        ],
    }
]

test_input_a = [
    [7, 0, 8, 8],
    [0, 9, 0, 8],
    [7, 0, 0, 0]
]

show_examples(training_examples_a)
show_grid(test_input_a, "Task A: test input")


### My prediction for Task A

- **What changes:**
- **What stays the same:**
- **My predicted test output (describe it or draw it on paper):**

In [ ]:
prediction_a = run_solver(
    training_examples_a,
    test_input_a,
    max_steps=1,
    max_attempts=120,
    seed=6101,
)


In [ ]:
#@title Check Task A

def check_task_a():
    expected = [
        [8, 8, 0, 7],
        [8, 0, 9, 0],
        [0, 0, 0, 7]
    ]
    assert prediction_a is not None, "Run the search after finishing all six exercises."
    assert prediction_a == expected, "The test prediction does not match. Inspect the discovered sequence."
    print("Task A solved: your program discovered a rule and applied it to a new input!")


make_check_button(check_task_a)


## Task B

Study both training examples and the test input before starting the search.

In [ ]:
training_examples_b = [
    {
        "input": [
            [1, 2, 0],
            [3, 0, 4]
        ],
        "output": [
            [3, 1],
            [0, 2],
            [4, 0]
        ],
    },
    {
        "input": [
            [5, 0],
            [6, 7],
            [0, 8]
        ],
        "output": [
            [0, 6, 5],
            [8, 7, 0]
        ],
    }
]

test_input_b = [
    [2, 0, 3, 0],
    [2, 4, 0, 5]
]

show_examples(training_examples_b)
show_grid(test_input_b, "Task B: test input")


### My prediction for Task B

- **What changes:**
- **What stays the same:**
- **My predicted test output (describe it or draw it on paper):**

In [ ]:
prediction_b = run_solver(
    training_examples_b,
    test_input_b,
    max_steps=2,
    max_attempts=1000,
    seed=6102,
)


In [ ]:
#@title Check Task B

def check_task_b():
    expected = [
        [2, 2],
        [4, 0],
        [0, 3],
        [5, 0]
    ]
    assert prediction_b is not None, "Run the search after finishing all six exercises."
    assert prediction_b == expected, "The test prediction does not match. Inspect the discovered sequence."
    print("Task B solved: your program discovered a rule and applied it to a new input!")


make_check_button(check_task_b)


## Task C

Study both training examples and the test input before starting the search.

In [ ]:
training_examples_c = [
    {
        "input": [
            [0, 0, 0, 0, 0, 0],
            [0, 0, 3, 0, 0, 0],
            [0, 3, 3, 3, 0, 0],
            [0, 0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0, 0]
        ],
        "output": [
            [4, 4, 4, 4, 4],
            [4, 0, 3, 0, 4],
            [4, 3, 3, 3, 4],
            [4, 4, 4, 4, 4]
        ],
    },
    {
        "input": [
            [0, 0, 0, 0, 0],
            [0, 2, 2, 0, 0],
            [0, 2, 0, 0, 0],
            [0, 2, 2, 0, 0],
            [0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0]
        ],
        "output": [
            [4, 4, 4, 4],
            [4, 2, 2, 4],
            [4, 2, 0, 4],
            [4, 2, 2, 4],
            [4, 4, 4, 4]
        ],
    }
]

test_input_c = [
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 8, 8, 0, 0, 0],
    [0, 0, 0, 8, 8, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0]
]

show_examples(training_examples_c)
show_grid(test_input_c, "Task C: test input")


### My prediction for Task C

- **What changes:**
- **What stays the same:**
- **My predicted test output (describe it or draw it on paper):**

In [ ]:
prediction_c = run_solver(
    training_examples_c,
    test_input_c,
    max_steps=2,
    max_attempts=1000,
    seed=6103,
)


In [ ]:
#@title Check Task C

def check_task_c():
    expected = [
        [4, 4, 4, 4, 4],
        [4, 8, 8, 0, 4],
        [4, 0, 8, 8, 4],
        [4, 4, 4, 4, 4]
    ]
    assert prediction_c is not None, "Run the search after finishing all six exercises."
    assert prediction_c == expected, "The test prediction does not match. Inspect the discovered sequence."
    print("Task C solved: your program discovered a rule and applied it to a new input!")


make_check_button(check_task_c)


# Optional challenge: Allow three steps

Task D needs a longer program than the core tasks. Keep the same functions and increase `max_steps` to `3`.

With six library choices, there are 6 possible one-step lists, 36 two-step lists, and 216 three-step lists. Your random search can revisit a candidate, and it chooses lengths uniformly, so it does not sample all 258 lists with equal probability.

The provided budget is larger for this task. Even so, the general meaning of `None` remains “no solution found within this search.”

## Challenge Task D

Study both training examples and the test input before starting the search.

In [ ]:
training_examples_d = [
    {
        "input": [
            [0, 0, 0, 0, 0, 0],
            [0, 1, 0, 3, 0, 0],
            [0, 1, 1, 3, 0, 0],
            [0, 0, 0, 0, 0, 0]
        ],
        "output": [
            [2, 2],
            [0, 2],
            [3, 3]
        ],
    },
    {
        "input": [
            [0, 0, 0, 0, 0],
            [0, 0, 1, 1, 0],
            [0, 0, 0, 3, 0],
            [0, 0, 1, 3, 0],
            [0, 0, 0, 0, 0]
        ],
        "output": [
            [2, 0, 2],
            [2, 3, 3]
        ],
    }
]

test_input_d = [
    [0, 0, 0, 0, 0, 0],
    [0, 0, 1, 3, 1, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0]
]

show_examples(training_examples_d)
show_grid(test_input_d, "Challenge Task D: test input")


### My prediction for Challenge Task D

- **What changes:**
- **What stays the same:**
- **My predicted test output (describe it or draw it on paper):**

In [ ]:
prediction_d = run_solver(
    training_examples_d,
    test_input_d,
    max_steps=3,
    max_attempts=4000,
    seed=6104,
)


In [ ]:
#@title Check Challenge Task D

def check_task_d():
    expected = [
        [2, 2],
        [3, 0],
        [2, 0]
    ]
    assert prediction_d is not None, "Run the search after finishing all six exercises."
    assert prediction_d == expected, "The test prediction does not match. Inspect the discovered sequence."
    print("Challenge Task D solved: your program discovered a rule and applied it to a new input!")


make_check_button(check_task_d)


## What you have built

Your program can propose a short program, check it against several examples, and use a successful program on a new input. That is an ARC-style solver for a family of tasks defined by this library.

It also has clear limits:

- A task may require an operation that is missing from the library.
- Its solution may need more steps than the current limit.
- Random sampling may miss a matching program within the budget.
- A program may fit the training examples but make the wrong prediction on a new input.

These are useful questions to investigate. The pieces you built—representation, candidate generation, testing, and search—give you a way to experiment with them.

## Optional experiments

1. **Change the seed:** Run one task with several seeds. Do the returned sequences differ? Do their outputs agree?
2. **Change the budget:** Try 1, 10, and 100 attempts. Why might a longer budget help without guaranteeing success?
3. **Investigate an ambiguous example:** Search using only the symmetric training pair above. Then test the result on the extra example.
4. **Extend the library:** Add one transformation and its name, then create a task that needs it.
5. **Build a puzzle factory:** Generate several inputs using Week 6(a)'s ideas. Choose one pipeline, use it to create every training output, and see whether your solver finds a matching program.

## Wrap-up and reflection

- **How a random list of names becomes a program:**
- **Why the same candidate must be tested on every training pair:**
- **What None means after a search:**
- **A case where more examples helped:**
- **A sequence my solver discovered:**
- **One improvement I would investigate next:**

## Share your completed notebook

When you finish:

1. Confirm that all six exercise checks pass.
2. Use your solver on Tasks A, B, and C.
3. Try Task D if you have time.
4. Complete the reflection.
5. In Colab, click **Share**.
6. Add **michael.d.moffitt@gmail.com** as a Viewer or Commenter.
7. Send the share invite.

You have connected what you know about functions, loops, lists, examples, and randomness into a program that searches for other programs.